# 04 — Regressão: clima → rendimento

Entradas: `crop_yield_limpo.csv` (treino) e `perfil_regiao.csv` (aplicação nas regiões de café). Os dois datasets não se unem por região+ano, então o modelo aprende a relação em `crop_yield` e é **transferido** ao clima das regiões de café. O resultado é uma estimativa, não um rendimento observado.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

crop = pd.read_csv('../data/processed/crop_yield_limpo.csv').sample(100000, random_state=42)
X = crop[['Rainfall_mm', 'Temperature_Celsius']]
y = crop['Yield_tons_per_hectare']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
len(X_train), len(X_test)

(80000, 20000)

## Comparação de modelos
Validação cruzada de 5 folds no treino (R²), com `StandardScaler` dentro do pipeline para a padronização usar só o treino de cada fold.

In [2]:
modelos = {
    'Regressão Linear': LinearRegression(),
    'Árvore (CART)': DecisionTreeRegressor(max_depth=8, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=15),
}
resultados = {}
for nome, m in modelos.items():
    pipe = make_pipeline(StandardScaler(), m)
    resultados[nome] = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    print(f'{nome}: R² {resultados[nome].mean():.3f} (±{resultados[nome].std():.3f})')

plt.boxplot(list(resultados.values()), tick_labels=list(resultados.keys()))
plt.ylabel('R² (validação cruzada)')
plt.savefig('../data/processed/comparacao_modelos.png', bbox_inches='tight')
plt.show()

Regressão Linear: R² 0.593 (±0.004)


Árvore (CART): R² 0.583 (±0.005)


KNN: R² 0.566 (±0.004)


C:\Users\ninja\AppData\Local\Temp\ipykernel_20632\3394602120.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Avaliação no conjunto de teste

In [3]:
escolhido = 'Regressão Linear'  # trocar se outro modelo vencer acima (a regressão é simples e interpretável)
modelo = make_pipeline(StandardScaler(), modelos[escolhido]).fit(X_train, y_train)
pred = modelo.predict(X_test)
print('MAE', round(mean_absolute_error(y_test, pred), 3), '| RMSE', round(mean_squared_error(y_test, pred) ** 0.5, 3), '| R²', round(r2_score(y_test, pred), 3))

MAE 0.878 | RMSE 1.079 | R² 0.593


## Aplicação nas 25 regiões de café
A chuva entra em mm/ano (`precipitacao_anual`), a mesma unidade do treino. Algumas regiões passam de 1000 mm/ano, fora da faixa vista no treino (100–1000 mm). Para não extrapolar, limitamos a entrada à faixa do treino e sinalizamos essas regiões.

In [4]:
perfis = pd.read_csv('../data/processed/perfil_regiao.csv')
entrada = perfis.rename(columns={'precipitacao_anual': 'Rainfall_mm', 'temperatura_media': 'Temperature_Celsius'})[['Rainfall_mm', 'Temperature_Celsius']].copy()

faixa = X_train.agg(['min', 'max'])
perfis['fora_da_faixa_treino'] = ((entrada < faixa.loc['min']) | (entrada > faixa.loc['max'])).any(axis=1)
entrada = entrada.clip(faixa.loc['min'], faixa.loc['max'], axis=1)

perfis['rendimento_estimado_ton_ha'] = modelo.predict(entrada)
perfis[['pais', 'regiao', 'precipitacao_anual', 'temperatura_media', 'fora_da_faixa_treino', 'rendimento_estimado_ton_ha']].sort_values('rendimento_estimado_ton_ha', ascending=False)

,pais,regiao,precipitacao_anual,temperatura_media,fora_da_faixa_treino,rendimento_estimado_ton_ha
23,Mexico,Chiapas,2534.031607,26.861679,True,6.883662
21,Honduras,Danli,1126.384364,23.907542,True,6.823993
22,Honduras,San Jose,1333.488952,23.565303,True,6.817080
10,Colombia,Morelia,1531.749552,23.269165,True,6.811098
20,Honduras,Campamento,1200.821295,23.124040,True,6.808167
7,Brazil,Sao Paulo Sao Jose,1338.725116,22.886925,True,6.803378
0,Brazil,Espirito Santo Afonso,1088.171063,22.758541,True,6.800784
15,Ethiopia,Bare,2218.489310,22.745291,True,6.800517
18,Guatemala,Jalapa,1242.011071,22.206865,True,6.789641
1,Brazil,Espirito Santo Conceicao,1150.006085,22.169075,True,6.788878


In [5]:
perfis.sort_values('rendimento_estimado_ton_ha', ascending=False).to_csv('../data/processed/estimativa_produtividade.csv', index=False)
perfis['fora_da_faixa_treino'].sum()

np.int64(23)